In [ ]:
import os, cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from PIL import Image


In [ ]:
train_dir="Intel-Image-Classification/train"
val_dir="Intel-Image-Classification/val"
test_dir="Intel-Image-Classification/test"
IMG_SIZE=(128,128)
BATCH_SIZE=32
EPOCHS=10


In [ ]:
train_datagen=ImageDataGenerator(rescale=1./255,rotation_range=20,zoom_range=0.2,horizontal_flip=True)
test_datagen=ImageDataGenerator(rescale=1./255)

train_gen=train_datagen.flow_from_directory(train_dir,target_size=IMG_SIZE,batch_size=BATCH_SIZE,class_mode='categorical')
val_gen=test_datagen.flow_from_directory(val_dir,target_size=IMG_SIZE,batch_size=BATCH_SIZE,class_mode='categorical')
test_gen=test_datagen.flow_from_directory(test_dir,target_size=IMG_SIZE,batch_size=BATCH_SIZE,class_mode='categorical',shuffle=False)


In [ ]:
class_counts={c:len(os.listdir(os.path.join(train_dir,c))) for c in os.listdir(train_dir)}
pd.DataFrame(class_counts.items(),columns=['Class','Count'])


In [ ]:
plt.figure(figsize=(8,4))
sns.barplot(x=list(class_counts.keys()),y=list(class_counts.values()))
plt.show()


In [ ]:
x,y=next(train_gen)
plt.figure(figsize=(12,8))
for i in range(12):
    plt.subplot(3,4,i+1)
    plt.imshow(x[i])
    plt.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
fnn_model=Sequential([
Flatten(input_shape=(128,128,3)),
Dense(512,activation='relu'),
Dense(256,activation='relu'),
Dense(6,activation='softmax')
])
fnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])


In [ ]:
cnn_model=Sequential([
Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
MaxPooling2D(),
Conv2D(64,(3,3),activation='relu'),
MaxPooling2D(),
Flatten(),
Dense(128,activation='relu'),
Dense(6,activation='softmax')
])
cnn_model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])


In [ ]:
deep_cnn=Sequential([
Conv2D(32,(3,3),activation='relu',input_shape=(128,128,3)),
BatchNormalization(),
MaxPooling2D(),
Conv2D(64,(3,3),activation='relu'),
BatchNormalization(),
MaxPooling2D(),
Conv2D(128,(3,3),activation='relu'),
BatchNormalization(),
MaxPooling2D(),
Dropout(0.3),
Flatten(),
Dense(256,activation='relu'),
Dropout(0.5),
Dense(6,activation='softmax')
])
deep_cnn.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])


In [ ]:
callbacks=[EarlyStopping(patience=3,restore_best_weights=True),ReduceLROnPlateau(patience=2)]


In [ ]:
fnn_history=fnn_model.fit(train_gen,validation_data=val_gen,epochs=EPOCHS,callbacks=callbacks)


In [ ]:
cnn_history=cnn_model.fit(train_gen,validation_data=val_gen,epochs=EPOCHS,callbacks=callbacks)


In [ ]:
deep_history=deep_cnn.fit(train_gen,validation_data=val_gen,epochs=EPOCHS,callbacks=callbacks)


In [ ]:
def evaluate_model(model,name):
    probs=model.predict(test_gen)
    preds=np.argmax(probs,axis=1)
    y_true=test_gen.classes
    acc=accuracy_score(y_true,preds)
    pre=precision_score(y_true,preds,average='weighted')
    rec=recall_score(y_true,preds,average='weighted')
    f1=f1_score(y_true,preds,average='weighted')
    return [name,acc,pre,rec,f1]

results=[]
results.append(evaluate_model(fnn_model,"FNN"))
results.append(evaluate_model(cnn_model,"CNN"))
results.append(evaluate_model(deep_cnn,"Deep CNN"))
comparison=pd.DataFrame(results,columns=["Model","Accuracy","Precision","Recall","F1"])
comparison


In [ ]:
plt.figure(figsize=(12,5))
plt.plot(deep_history.history['accuracy'])
plt.plot(deep_history.history['val_accuracy'])
plt.show()


In [ ]:
plt.figure(figsize=(12,5))
plt.plot(deep_history.history['loss'])
plt.plot(deep_history.history['val_loss'])
plt.show()


In [ ]:
pred_probs=deep_cnn.predict(test_gen)
preds=np.argmax(pred_probs,axis=1)
cm=confusion_matrix(test_gen.classes,preds)
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt='d')
plt.show()
print(classification_report(test_gen.classes,preds))


In [ ]:
deep_cnn.save("best_model.keras")
best_model=load_model("best_model.keras")


In [ ]:
class_names=list(train_gen.class_indices.keys())
paths=test_gen.filepaths[:5]

plt.figure(figsize=(15,8))
for i,p in enumerate(paths):
    img=tf.keras.utils.load_img(p,target_size=IMG_SIZE)
    arr=tf.keras.utils.img_to_array(img)/255.0
    pred=np.argmax(best_model.predict(np.expand_dims(arr,axis=0),verbose=0))
    true_label=p.split(os.sep)[-2]
    plt.subplot(1,5,i+1)
    plt.imshow(img)
    plt.title(f"T:{true_label}\nP:{class_names[pred]}")
    plt.axis('off')
plt.show()
